In [11]:
import pandas as pd
import json
import os
import random
from openai import OpenAI #estamos la clase concreta OpenAI del módulo openai
from dotenv import load_dotenv #importamos una función concreta del módulo
load_dotenv("template.env")

True

In [12]:
# Acceder a la clave de API de OpenAI
api_key = os.getenv("OPENAI_API_KEY")

# Asegurarte de que la clave de API se haya cargado correctamente
if api_key is None:
	raise ValueError("La clave de API no está configurada en el archivo .env")

client = OpenAI() #creando un objeto de la clase"

In [13]:
dataset_folder = os.getenv("DATASET_FOLDER")
dataset_path = str(dataset_folder) + "Alonso_2014_SpanishAoA.xlsx"

df = pd.read_excel(dataset_path)
#df = df.sample(100)
df.head()

c:\Users\Eneko\Documents\UniUPM\BECA\Lenguaje_interpret\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,word,averageAoA,SD,Min,Max,ItemZScore,OralFreq_Log,WrittenFreq_Subtlex-ESP_Log,WrittenFreq_LEXESP_Log,WrittenFreq_espal_Log,espal_max_lem_cat,max_lem_code,espal_es_num_syll
0,a,2.28,1.443352,1,6,-1.650147,4.851582,5.984858,4.960556,4.358684,ADPOSITION,SPS00,1.0
1,abajo,2.96,1.369642,1,6,-1.356683,2.615950,3.940915,2.750508,1.811229,ADVERB,RG,3.0
2,abandonado,6.06,1.658743,2,10,-0.584383,1.623249,2.872156,2.195900,1.374186,ADJECTIVE,AQ0MSP,5.0
3,abandonar,7.58,1.654801,4,11,-0.019940,1.806180,2.987666,2.434569,1.653307,VERB,VMN0000,4.0
4,abandono,7.22,1.940860,3,11,0.040498,1.602060,2.336460,2.167317,1.328129,NOUN,NCMS000,4.0


In [14]:
#FUNCTION DECLARATION

def word_into_prompt(prompt,word):
	return prompt.replace("{palabra}",word)

def word_aoa_into_answer(word,aoa):
	return json.dumps({"Word": str(word) , "AoA" : str(aoa)})

def generate_fine_tuning(prompt,word,aoa):
	f_t_line = {
		"messages": [
			{ "role": "user", "content": word_into_prompt(prompt,word) },
			{ "role": "assistant", "content": word_aoa_into_answer(word,aoa) }
		]}
	return f_t_line

def create_file_from_tasks(tasks,file_name):
	with open(file_name, 'w') as file:
		for obj in tasks:
			file.write(json.dumps(obj) + '\n')


def create_fine_tunning_from_json(json_object,prompt):
	word = json_object["word"]
	aoa = json_object["averageAoA"]
	f_t_line = generate_fine_tuning(
		prompt,word,aoa
		)
	return f_t_line


def create_f_t_array_from_dataframe(df,prompt):
	tasks = []
	for index, row in df.iterrows():
		task = create_fine_tunning_from_json(row,prompt)
		tasks.append(task)
	return tasks


def get_line_file(file_name,line,extract_func):
	with open(file_name, 'r') as f:
		for line_number, theline in enumerate(f):
			if line_number == line:
				res = theline
				break
	res = json.loads(res)
	return extract_func(res)


def extract_input(new_line):
	return (new_line["messages"])


def upload_file(file_name: str, purpose: str) -> str:
    with open(file_name, "rb") as file_fd:
        response = client.files.create(file=file_fd, purpose=purpose)
    return response.id

In [15]:
#PROMPTS

#AGE PROMPT
categorize_system_prompt_paraphrase = '''
La edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez.
En concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito.
Estima la edad media de adquisición (AoA) de la palabra "{palabra}" para un hablante nativo de español.
El formato de salida debe ser un objeto JSON. Por ejemplo: { Word: {palabra} , AoA: }
'''

In [16]:
#SET output folder
output_folder = os.getenv("OUTPUT_FOLDER")
def out_file(file_name): return (str(output_folder) + file_name)

middle_folder = os.getenv("MIDDLE_FOLDER")
def middle_file(file_name): return (str(middle_folder) + file_name)

In [17]:
f_t_file_array = [middle_file("batch_job_mmlu_f_t_aoa_alonso.jsonl"),
				  middle_file("batch_job_mmlu_check_aoa_alonso.jsonl"),
				  middle_file("batch_job_mmlu_batch_aoa_alonso.jsonl")]

In [18]:
#CREATE tasks from the database
f_t_array = [create_f_t_array_from_dataframe(df,categorize_system_prompt_paraphrase)]

In [19]:
#SEPARATE words in piles to fine-tune or check
num_task_train = 2000

#check all tasks in task array
max_num_task = 0
for i in range(0,len(f_t_array)):
	max_num_task += len(f_t_array[i])

#generate random task indexes
random_num_array = []
for i in range(0,num_task_train):
	new_num = random.randrange(0,max_num_task)
	while(new_num in random_num_array):
		new_num = random.randrange(0,max_num_task)
	random_num_array.append(new_num)

#create training array and check array
indiv = [[] for Null in range(len(f_t_file_array))]

index_com = 0
for i in range(0,len(f_t_array)):
	tasks = f_t_array[i]
	for j in range(0,len(tasks)):
		if ((index_com+j)in random_num_array):
			indiv[0].append(tasks[j])
		else:
			indiv[1].append(tasks[j])
	index_com += len(f_t_array[i])

#create files
for i in range(0,len(indiv)):
	create_file_from_tasks(indiv[i],f_t_file_array[i])

In [20]:
#TRAIN fine tuning file
training_file_id = upload_file(f_t_file_array[0], "fine-tune")

job = client.fine_tuning.jobs.create(
		training_file = training_file_id,
		model = "gpt-4o-mini-2024-07-18",
	)

In [26]:
#CHECK fine tuning progress
f_t_job = client.fine_tuning.jobs.retrieve(job.id)
print(f_t_job)
print(f_t_job.status)
print(f_t_job.id)

FineTuningJob(id='ftjob-Y108Q6Dahzzq6zPyEuFQZylS', created_at=1744289467, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-b9e6eTH4lj1kn4VhwdRfE1Rx', result_files=[], seed=881744774, status='validating_files', trained_tokens=None, training_file='file-1UkRhukhDCyGXDnZ7StA1M', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None)
validating_files
ftjob-Y108Q6Dahzzq6zPyEuFQZylS


In [ ]:
#EXTRACT fine tuned model
ft_job_id = ""
f_t_job = client.fine_tuning.jobs.retrieve(ft_job_id)
#ft_job_id = job.id

fine_tuned_model_id = f_t_job.fine_tuned_model
print(f_t_job)
print(fine_tuned_model_id)

FineTuningJob(id='ftjob-AlcxNvpP7kDLbTugyDRxDwoc', created_at=1744220117, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:ging-upm::BKU95Fsm', finished_at=1744222138, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-b9e6eTH4lj1kn4VhwdRfE1Rx', result_files=['file-PSEBRLknyqF4fqeU8iHC6k'], seed=77590528, status='succeeded', trained_tokens=818532, training_file='file-CTq7vPUs3dcW3pT3nMaBQm', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None)
ft:gpt-4o-mini-2024-07-18:ging-upm::BKU95Fsm


In [27]:
def generate_task(index,model,mess_content):
	task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "temperature": 0,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [mess_content],
        }
    }
	return task

def create_batch(file_name):
	batch_file = client.files.create(
		file = open(file_name, "rb"),
		purpose = "batch"
	)
	batch_job = client.batches.create(
		input_file_id = batch_file.id,
		endpoint = "/v1/chat/completions",
		completion_window = "24h"
	)
	return batch_job

In [ ]:
#GENERATE BATCH FILE
test_file = f_t_file_array[1]
with open(test_file, 'r') as f:
	lines = len(f.readlines())

print("[lines]: "+str(lines))

batch_job_tasks = []
for i in range(0,lines):
	line = get_line_file(test_file,i,extract_input)
	task = generate_task(i,fine_tuned_model_id,line[0])
	batch_job_tasks.append(task)

create_file_from_tasks(batch_job_tasks,f_t_file_array[2])

[lines]: 5039


In [ ]:
#GENERATE BATCH
ba_jo = create_batch(f_t_file_array[2])

In [35]:
#COMPLETION CHECK
batch = client.batches.retrieve(ba_jo.id)
result_file_id = batch.output_file_id
status = batch.status
print(batch)
print(status)

Batch(id='batch_67f6df5dfc008190aa37ffa335982020', completion_window='24h', created_at=1744232285, endpoint='/v1/chat/completions', input_file_id='file-4d5pSvPrH213yYNAMVj5J5', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1744234801, error_file_id=None, errors=None, expired_at=None, expires_at=1744318685, failed_at=None, finalizing_at=1744234309, in_progress_at=1744232289, metadata=None, output_file_id='file-CFFR7CLwwjogGkBybECvyK', request_counts=BatchRequestCounts(completed=5039, failed=0, total=5039))
completed


In [38]:
#OUTPUT FILE
batch = client.batches.retrieve(batch.id)
result_file_id = batch.output_file_id

result = client.files.content(result_file_id).content

result_file_name = f_t_file_array[2].replace(".json","_result.json")
result_file_name = result_file_name.replace("middle_files","output_files")

with open(result_file_name, 'wb') as file:
	file.write(result)

In [13]:
def extract_data(new_line):
	res = new_line["response"]["body"]["choices"][0]["message"]["content"]
	res = json.loads(res)
	return res

out_file_name = "output_files/New_Results_AoA_f_t_2000.xlsx"
file_name1 = "middle_files/batch_job_mmlu_check_aoa_alonso.jsonl"
file_name2 = "output_files/batch_job_mmlu_batch_aoa_alonso_result.jsonl"

rows = []
errors = []

with open(file_name1, 'r') as f:
	lines = len(f.readlines())

for i in range(0,lines):
	line1 = get_line_file(file_name1,i,extract_input)
	line1 = json.loads(line1[1]["content"])
	try:
		line2 = get_line_file(file_name2,i,extract_data)
		#assert(line1["Word"]==line2["Word"])
		newAoA = line2["AoA"]
	except:
		newAoA = "NaN"
		print(f"line_num {i} \nline {line1}")
		errors.append({"Line_Num":i})
	rows.append({
  	  	"Word":line1['Word'],
		"FT_AoA":newAoA,
		"not_FT_AoA":line1["AoA"],
	})

clean_dtset,errors_dtset = pd.DataFrame(rows), pd.DataFrame(errors)

with pd.ExcelWriter(out_file_name) as writer:
	clean_dtset.to_excel(writer, sheet_name='Results',index=False)
	errors_dtset.to_excel(writer, sheet_name='Errors',index=False)

line_num 45 
line {'Word': 'ácido', 'AoA': '7.78'}
line_num 79 
line {'Word': 'adiós', 'AoA': '1.74'}
line_num 131 
line {'Word': 'águila', 'AoA': '5.22'}
line_num 339 
line {'Word': 'ártico', 'AoA': '8.38'}
line_num 386 
line {'Word': 'átomo', 'AoA': '10.44'}
line_num 1728 
line {'Word': 'éramos', 'AoA': '5.92'}
line_num 1771 
line {'Word': 'ése', 'AoA': '4.04'}
line_num 1844 
line {'Word': 'éstas', 'AoA': '5.18'}
line_num 2278 
line {'Word': 'hábito', 'AoA': '8.94'}
line_num 2478 
line {'Word': 'índice', 'AoA': '8.3'}
line_num 2805 
line {'Word': 'lóbrego', 'AoA': '10.8'}
line_num 3003 
line {'Word': 'mérito', 'AoA': '9.5'}
line_num 3349 
line {'Word': 'óptica', 'AoA': '8.38'}
line_num 3350 
line {'Word': 'óptico', 'AoA': '8.36'}


In [ ]:
#SAVE ouput in a .xlsx file
file_name = out_file("New_Results_AoA_f_t_2000.xlsx")
clean_dtset = pd.DataFrame(rows)

#CALCULATE correlation coeff

cmp_clm_1 = "not_FT_AoA"
cmp_clm_2 = "FT_AoA"

pearson_corr = clean_dtset[[cmp_clm_1, cmp_clm_2]].corr('pearson')
pearson_corr = pearson_corr[cmp_clm_1][cmp_clm_2]
print(f"Pearson_Corr: {pearson_corr}")

spearman_corr = clean_dtset[[cmp_clm_1, cmp_clm_2]].corr('spearman')
spearman_corr = spearman_corr[cmp_clm_1][cmp_clm_2]
print(f"Spearman_Corr: {spearman_corr}")


Pearson_Corr: 0.9321803616703224
Spearman_Corr: 0.9325337109395209
